# SEN1Floods11 — U-Net Training (Google Colab)

**Runtime:** GPU (Runtime → Change runtime type → T4 GPU)

**First run:** Cells 1–4 set up the environment and download data to Google Drive (~700 MB, takes ~2 min). Subsequent runs skip the download.

**Workflow:**
1. Check GPU
2. Mount Google Drive
3. Clone repo + install dependencies
4. Download data (first run only)
5. Train U-Net
6. Evaluate on test set

## 1. Check GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime → Change runtime type → T4 GPU, then reconnect."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

## 2. Mount Google Drive

Data and checkpoints are stored in `My Drive/sen1flood11/` so they persist across sessions.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path

DRIVE_DIR   = Path("/content/drive/MyDrive/sen1flood11")
DATA_DIR    = DRIVE_DIR / "data/raw"
CKPT_DIR    = DRIVE_DIR / "checkpoints"
REPO_DIR    = Path("/content/sen1flood11")

DRIVE_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Drive mounted. Working directory: {DRIVE_DIR}")

## 3. Clone Repo and Install Dependencies

In [ ]:
import subprocess

# Clone or update repo
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    print("Repo updated.")
else:
    subprocess.run(
        ["git", "clone", "https://github.com/efeakp/sen1flood11.git", str(REPO_DIR)],
        check=True,
    )
    print("Repo cloned.")

# Add src/ to Python path
import sys
sys.path.insert(0, str(REPO_DIR / "src"))

# Install dependencies not pre-installed on Colab
!pip install -q segmentation-models-pytorch rasterio albumentations

## 4. Download Data (First Run Only)

Downloads the hand-labeled subset (~700 MB) from Google Cloud Storage to your Drive.
Skipped automatically on subsequent runs.

In [ ]:
S1_DIR = DATA_DIR / "S1Hand"

if S1_DIR.exists() and len(list(S1_DIR.glob("*.tif"))) == 446:
    print(f"Data already downloaded ({len(list(S1_DIR.glob('*.tif')))} S1 chips found). Skipping.")
else:
    print("Downloading SEN1Floods11 hand-labeled subset from GCS...")
    !gsutil -m rsync -r gs://sen1floods11/v1.1/data/flood_events/HandLabeled {DATA_DIR}
    !gsutil -m cp "gs://sen1floods11/v1.1/splits/flood_handlabeled/*.csv" {DATA_DIR}
    print("\nDownload complete.")

# Verify
for subdir in ["S1Hand", "LabelHand"]:
    n = len(list((DATA_DIR / subdir).glob("*.tif")))
    print(f"  {subdir}: {n} chips")

## 5. Configure Training

In [ ]:
config = {
    "data": {
        "raw_dir":   str(DATA_DIR),
        "train_csv": str(DATA_DIR / "flood_train_data.csv"),
        "val_csv":   str(DATA_DIR / "flood_valid_data.csv"),
        "test_csv":  str(DATA_DIR / "flood_test_data.csv"),
        "in_channels": 2,
    },
    "model": {
        "encoder":          "resnet34",
        "encoder_weights":  "imagenet",
        "num_classes":       1,
    },
    "training": {
        "epochs":                   50,
        "batch_size":               16,   # fits comfortably on T4 16 GB
        "learning_rate":            1e-4,
        "weight_decay":             1e-5,
        "early_stopping_patience": 10,
    },
    "logging": {
        "checkpoint_dir": str(CKPT_DIR),
    },
}

print("Config:")
for section, vals in config.items():
    print(f"  {section}: {vals}")

## 6. Train

In [ ]:
import torch
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import albumentations as A

from dataset import SEN1FloodsDataset
from model import build_model, get_loss
from evaluate import evaluate_batch

device = torch.device("cuda")

# Augmentations for training
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
])

train_ds = SEN1FloodsDataset(
    split_csv=config["data"]["train_csv"],
    data_dir=config["data"]["raw_dir"],
    transform=train_transform,
)
val_ds = SEN1FloodsDataset(
    split_csv=config["data"]["val_csv"],
    data_dir=config["data"]["raw_dir"],
)

train_loader = DataLoader(train_ds, batch_size=config["training"]["batch_size"],
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=config["training"]["batch_size"],
                          shuffle=False, num_workers=2)

model = build_model(
    encoder=config["model"]["encoder"],
    encoder_weights=config["model"]["encoder_weights"],
    in_channels=config["data"]["in_channels"],
    num_classes=config["model"]["num_classes"],
).to(device)

criterion = get_loss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["training"]["learning_rate"],
    weight_decay=config["training"]["weight_decay"],
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=config["training"]["epochs"]
)
scaler = torch.cuda.amp.GradScaler()  # mixed precision

ckpt_path = Path(config["logging"]["checkpoint_dir"]) / "best_model.pth"
best_iou, patience_counter = 0.0, 0
history = {"train_loss": [], "val_iou": [], "val_f1": []}

print(f"Training on {len(train_ds)} chips | Validating on {len(val_ds)} chips")
print(f"Checkpoints → {ckpt_path}\n")

for epoch in range(1, config["training"]["epochs"] + 1):
    # --- Train ---
    model.train()
    train_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch:02d} train", leave=False):
        images = batch["image"].to(device)
        masks  = batch["mask"].to(device).unsqueeze(1).float()

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(images)
            loss   = criterion(logits, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    # --- Validate ---
    model.eval()
    val_iou, val_f1 = 0.0, 0.0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch:02d} val  ", leave=False):
            images = batch["image"].to(device)
            masks  = batch["mask"].to(device).unsqueeze(1).float()
            with torch.cuda.amp.autocast():
                logits = model(images)
            m = evaluate_batch(logits.float(), masks)
            val_iou += m["iou"]
            val_f1  += m["f1"]

    train_loss /= len(train_loader)
    val_iou    /= len(val_loader)
    val_f1     /= len(val_loader)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_iou"].append(val_iou)
    history["val_f1"].append(val_f1)

    flag = ""
    if val_iou > best_iou:
        best_iou = val_iou
        patience_counter = 0
        torch.save(model.state_dict(), ckpt_path)
        flag = "  ← saved"
    else:
        patience_counter += 1

    print(f"Epoch {epoch:02d} | loss={train_loss:.4f} | val_iou={val_iou:.4f} | val_f1={val_f1:.4f}{flag}")

    if patience_counter >= config["training"]["early_stopping_patience"]:
        print(f"Early stopping at epoch {epoch}.")
        break

print(f"\nTraining complete. Best val IoU: {best_iou:.4f}")
print(f"Checkpoint saved to: {ckpt_path}")

## 7. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, history["train_loss"], label="Train loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")
axes[0].legend()

axes[1].plot(epochs, history["val_iou"], label="Val IoU")
axes[1].plot(epochs, history["val_f1"],  label="Val F1")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].set_title("Validation Metrics")
axes[1].legend()

plt.tight_layout()
plt.savefig(str(DRIVE_DIR / "training_curves.png"), bbox_inches="tight")
plt.show()
print(f"Saved to {DRIVE_DIR / 'training_curves.png'}")

## 8. Evaluate on Test Set

In [ ]:
from evaluate import evaluate_batch

test_ds = SEN1FloodsDataset(
    split_csv=config["data"]["test_csv"],
    data_dir=config["data"]["raw_dir"],
)
test_loader = DataLoader(test_ds, batch_size=config["training"]["batch_size"],
                         shuffle=False, num_workers=2)

# Load best checkpoint
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.eval()

test_iou, test_f1 = 0.0, 0.0
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        images = batch["image"].to(device)
        masks  = batch["mask"].to(device).unsqueeze(1).float()
        with torch.cuda.amp.autocast():
            logits = model(images)
        m = evaluate_batch(logits.float(), masks)
        test_iou += m["iou"]
        test_f1  += m["f1"]

test_iou /= len(test_loader)
test_f1  /= len(test_loader)

print("\n=== Test Set Results ===")
print(f"  IoU : {test_iou:.4f}")
print(f"  F1  : {test_f1:.4f}")

## 9. Visualise Predictions

In [ ]:
import numpy as np

model.eval()
batch = next(iter(test_loader))
images = batch["image"].to(device)
masks  = batch["mask"]

with torch.no_grad(), torch.cuda.amp.autocast():
    logits = model(images)
preds = (torch.sigmoid(logits.float()) > 0.5).squeeze(1).cpu().numpy()

n_show = min(4, len(images))
fig, axes = plt.subplots(n_show, 3, figsize=(11, n_show * 3.2))
fig.suptitle("Test predictions: VV input | Ground truth | Prediction", fontsize=12)

for i in range(n_show):
    vv   = images[i, 0].cpu().numpy()
    gt   = masks[i].numpy()
    pred = preds[i]

    axes[i, 0].imshow(vv,   cmap="gray")
    axes[i, 0].set_title("VV (normalised)", fontsize=9)
    axes[i, 1].imshow(gt,   cmap="Blues", vmin=0, vmax=1)
    axes[i, 1].set_title("Ground truth", fontsize=9)
    axes[i, 2].imshow(pred, cmap="Blues", vmin=0, vmax=1)
    axes[i, 2].set_title("Prediction", fontsize=9)
    for ax in axes[i]: ax.axis("off")

plt.tight_layout()
plt.savefig(str(DRIVE_DIR / "test_predictions.png"), bbox_inches="tight")
plt.show()
print(f"Saved to {DRIVE_DIR / 'test_predictions.png'}")